In [4]:
# main.py
# Pipeline complet pour le projet ARCHE (ETL -> features -> modèles -> évaluation)

from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --------------------
# 1. Chargement & ETL
# --------------------

class LogLoader:
    def __init__(self, log_path: str):
        self.log_path = Path(log_path)
        self.logs_df: pd.DataFrame | None = None

    def load(self) -> pd.DataFrame:
        """Charger le fichier logs_info_25_pseudo.csv"""
        self.logs_df = pd.read_csv(self.log_path)
        return self.logs_df

    def clean(self) -> pd.DataFrame:
        """Nettoyage simple des logs."""
        df = self.logs_df.copy()
        # Exemple : suppression des lignes avec pseudo manquant
        df = df.dropna(subset=["pseudo"])
        # TODO : autres nettoyages éventuels
        self.logs_df = df
        return self.logs_df


class GradeLoader:
    def __init__(self, grade_path: str):
        self.grade_path = Path(grade_path)
        self.grades_df: pd.DataFrame | None = None

    def load(self) -> pd.DataFrame:
        """Charger le fichier notes_info_25_pseudo.csv"""
        self.grades_df = pd.read_csv(self.grade_path)
        return self.grades_df

    def clean(self) -> pd.DataFrame:
        """Nettoyage simple des notes."""
        df = self.grades_df.copy()
        df = df.dropna(subset=["pseudo", "note"])
        self.grades_df = df
        return self.grades_df


class DatasetBuilder:
    def __init__(self, logs_df: pd.DataFrame, grades_df: pd.DataFrame):
        self.logs_df = logs_df
        self.grades_df = grades_df
        self.dataset: pd.DataFrame | None = None

    def merge_on_pseudo(self) -> pd.DataFrame:
        """Jointure logs/notes sur la colonne 'pseudo'."""
        df = self.logs_df.merge(self.grades_df, on="pseudo", how="inner")
        self.dataset = df
        return self.dataset


# --------------------
# 2. Feature engineering
# --------------------

class FeatureEngineer:
    def __init__(self, dataset: pd.DataFrame):
        self.dataset = dataset
        self.features_df: pd.DataFrame | None = None

    def create_basic_features(self) -> pd.DataFrame:
        """
        Exemple de features par pseudo :
        - nb_events : nombre total de lignes de logs
        - nb_contexts : nombre de contextes distincts
        - nb_composants : nombre de composants distincts
        - nb_evenements_types : nombre de types d'événements distincts
        """
        logs_cols = ["pseudo", "contexte", "composant", "evenement"]
        for col in logs_cols:
            if col not in self.dataset.columns:
                raise ValueError(f"Colonne manquante : {col}")

        # agrégation par pseudo
        agg = (
            self.dataset.groupby("pseudo")
            .agg(
                nb_events=("evenement", "count"),
                nb_contexts=("contexte", "nunique"),
                nb_composants=("composant", "nunique"),
                nb_evenements_types=("evenement", "nunique"),
            )
            .reset_index()
        )

        # récupérer la note (une valeur par pseudo)
        notes = self.dataset[["pseudo", "note"]].drop_duplicates()

        features = agg.merge(notes, on="pseudo", how="left")
        self.features_df = features
        return self.features_df

    def select_best_features(self) -> pd.DataFrame:
        """
        Pour l'instant, on garde toutes les features créées.
        C'est ici que tu pourras faire de la sélection (corrélations, importance, etc.).
        """
        return self.features_df


# --------------------
# 3. Modèles
# --------------------

class BaseModel:
    def __init__(self, name: str):
        self.name = name
        self.model = None
        self.metrics: dict | None = None

    def build_model(self):
        raise NotImplementedError

    def train(self, X: pd.DataFrame, y: pd.Series):
        self.build_model()
        self.model.fit(X, y)

    def predict(self, X: pd.DataFrame):
        return self.model.predict(X)


class RegressionModel(BaseModel):
    def __init__(self):
        super().__init__(name="LinearRegression")

    def build_model(self):
        self.model = LinearRegression()


class MLModel(BaseModel):
    def __init__(self):
        super().__init__(name="RandomForestRegressor")

    def build_model(self):
        self.model = RandomForestRegressor(
            n_estimators=100,
            random_state=42,
        )


# --------------------
# 4. Évaluation
# --------------------

class Evaluator:
    def __init__(self):
        self.results = []

    def evaluate_regression(self, model: BaseModel, X_test, y_test):
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        metrics = {"MSE": mse, "R2": r2}
        self.results.append({"model": model.name, **metrics})
        return metrics

    def compare_models(self) -> pd.DataFrame:
        return pd.DataFrame(self.results)


# --------------------
# 5. Pipeline complète
# --------------------

class Pipeline:
    def __init__(self, log_path: str, grade_path: str):
        self.log_loader = LogLoader(log_path)
        self.grade_loader = GradeLoader(grade_path)
        self.dataset_builder = None
        self.feature_engineer = None
        self.evaluator = Evaluator()

    def run(self) -> pd.DataFrame:
        # 1. ETL
        logs = self.log_loader.load()
        logs = self.log_loader.clean()

        grades = self.grade_loader.load()
        grades = self.grade_loader.clean()

        self.dataset_builder = DatasetBuilder(logs, grades)
        dataset = self.dataset_builder.merge_on_pseudo()

        # 2. Features
        self.feature_engineer = FeatureEngineer(dataset)
        features = self.feature_engineer.create_basic_features()
        features = self.feature_engineer.select_best_features()

        # 3. Séparation X, y
        if "note" not in features.columns:
            raise ValueError("La colonne 'note' est absente des features.")
        y = features["note"]
        X = features.drop(columns=["note", "pseudo"])

        # 4. Split train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # 5. Modèles
        reg_model = RegressionModel()
        reg_model.train(X_train, y_train)
        self.evaluator.evaluate_regression(reg_model, X_test, y_test)

        ml_model = MLModel()
        ml_model.train(X_train, y_train)
        self.evaluator.evaluate_regression(ml_model, X_test, y_test)

        # 6. Résultats
        results_df = self.evaluator.compare_models()
        return results_df


# --------------------
# 6. Point d'entrée
# --------------------

if __name__ == "__main__":
    # Adapter les chemins si besoin
    LOG_PATH = "logs_info_25_pseudo.csv"
    NOTES_PATH = "notes_info_25_pseudo.csv"

    pipeline = Pipeline(LOG_PATH, NOTES_PATH)
    results = pipeline.run()
    print("Résultats des modèles :")
    print(results)


FileNotFoundError: [Errno 2] No such file or directory: 'logs_info_25_pseudo.csv'